# 🚀 50M Bengali GPT - Full Training from Scratch (100% Unfrozen)
### ⚡ হাইলাইট ও স্পিড অপটিমাইজেশন:
- **প্যারামিটার:** ~54.3 Million (১০০% আনফ্রোজেন, প্রতিটি নিউরন স্ক্র্যাচ থেকে ভাষা শিখবে)
- **ভোকাবুলারি:** 10,000 (ByteLevel BPE, ~1.85 tokens/word - অক্ষরের বদলে আস্ত শব্দ শিখবে)
- **কনটেক্সট লেন্থ:** 2048 Tokens (~1,100 বাংলা শব্দ ধারণক্ষমতা)
- **স্পিড বুস্ট:** PyTorch SDPA (FlashAttention) + AMP FP16 + PyTorch 2.0 `torch.compile` (সর্বোচ্চ গতি!)
- **হার্ডওয়্যার:** Colab Free T4 GPU (~1.2 GB VRAM খরচ, 14 GB মেমরি নিরাপদ থাকবে)

In [ ]:
# Step 1: GPU চেক করুন (NVIDIA T4 থাকা নিশ্চিত করুন)
!nvidia-smi

In [ ]:
# Step 2: Google Drive মাউন্ট করুন (চেকপয়েন্ট সেভ করার জন্য)
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"✓ Drive Checkpoint ডিরেক্টরি প্রস্তুত: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# Step 3: রিপোজিটরি ক্লোন ও লাইব্রেরি ইনস্টল করুন
import os

%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git

%cd /content/ss_100m/ss_50million
!pip install -q tokenizers torch numpy
print("✓ এনভায়রনমেন্ট ও ডিপেনডেন্সি প্রস্তুত!")

In [ ]:
# Step 4: নতুন 10,000 ভোকাব টোকেনাইজার পরীক্ষা করুন
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("tokenizer.json")
print(f"✓ টোকেনাইজার ভোকাব সাইজ: {tokenizer.get_vocab_size():,}")

sample = "ডিজিটাল মার্কেটিং হলো আধুনিক যুগের ব্যবসার মূল ভিত্তি।"
enc = tokenizer.encode(sample)
words = sample.split()
print(f"টেক্সট: {sample}")
print(f"মোট শব্দ: {len(words)} টি | টোকেন: {len(enc.ids)} টি")
print(f"প্রতি শব্দে টোকেন অনুপাত: {len(enc.ids)/len(words):.2f} (গতি দ্বিগুণ বৃদ্ধি পেয়েছে!)")

In [ ]:
# Step 5: মডেল কনফিগারেশন যাচাই করুন
from src.config import GPTConfig
from src.model import BengaliGPT as GPT

p = GPTConfig.estimate_parameters()
print(f"মডেল মোট প্যারামিটার: {p['total_untied']/1e6:.2f}M")
print(f"কনটেক্সট লেন্থ: {GPTConfig.block_size} টোকেন")
print(f"মাইক্রো-ব্যাচ: {GPTConfig.batch_size}, একিউমুলেশন: {GPTConfig.gradient_accumulation_steps}")

In [ ]:
# Step 6: 🚀 প্রি-ট্রেনিং শুরু (স্ক্র্যাচ থেকে পুরো মডেল আনফ্রোজেন + সর্বোচ্চ গতি অপটিমাইজেশন)
import os
import sys
import time
import math
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

# ১. ডিভাইস ও সর্বোচ্চ GPU পারফরম্যান্স ফ্ল্যাগ
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True  # GPU কার্নেল অপটিমাইজেশন
    torch.backends.cuda.matmul.allow_tf32 = True  # ম্যাট্রিক্স মাল্টিপ্লিকেশন গতি বৃদ্ধি
    torch.backends.cudnn.allow_tf32 = True

# ২. টোকেনাইজার ও ডেটাসেট লোড
tokenizer = Tokenizer.from_file("tokenizer.json")
corpus_file = "data/corpus.txt"
dataset = BengaliDataset(corpus_path=corpus_file, tokenizer=tokenizer, block_size=GPTConfig.block_size)
print("✓ ডেটাসেট প্রস্তুত!")

# ৩. মডেল তৈরি (Full Scratch Initialization)
raw_model = GPT(GPTConfig).to(device)
total_params = sum(p.numel() for p in raw_model.parameters())
trainable_params = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
print(f"✓ মডেল সফলভাবে তৈরি হয়েছে!")
print(f"  - মোট প্যারামিটার: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"  - ট্রেইনেবল প্যারামিটার: {trainable_params:,} (১০০% আনফ্রোজেন, প্রতিটি নিউরন শিখবে!)")

# ৪. PyTorch 2.0 কম্পাইলেশন (গতি আরও ২০-৩০% বৃদ্ধির জন্য)
try:
    model = torch.compile(raw_model)
    print("✓ PyTorch 2.0 torch.compile সক্রিয় করা হয়েছে (সর্বোচ্চ স্পিড)!")
except Exception as e:
    print("ℹ️ torch.compile এড়িয়ে সাধারণ মোডে চলছে:", e)
    model = raw_model

# ৫. অপটিমাইজার ও শিডিউলার (Cosine Warmup)
optimizer = torch.optim.AdamW(raw_model.parameters(), lr=GPTConfig.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()

def get_lr(it, max_iters=5000, warmup_iters=250, lr=3e-4, min_lr=3e-5):
    if it < warmup_iters:
        return lr * it / warmup_iters
    if it > max_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (lr - min_lr)

# ৬. ট্রেনিং লুপ
print("=" * 60)
print("🔥 পূর্ণাঙ্গ প্রি-ট্রেনিং শুরু হচ্ছে (প্রতি ৫০০ স্টেপে ড্রাইভ ব্যাকআপ)... ")
print("=" * 60)

start_time = time.time()
model.train()
optimizer.zero_grad(set_to_none=True)

for step in range(1, GPTConfig.max_iters + 1):
    current_lr = get_lr(step, max_iters=GPTConfig.max_iters)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    accum_loss = 0.0
    for micro_step in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            logits, loss = model(x, y)
            loss = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if step % 50 == 0 or step == 1:
        elapsed = time.time() - start_time
        speed = step / elapsed if elapsed > 0 else 0
        print(f"Step {step:4d}/{GPTConfig.max_iters} | Loss: {accum_loss:.4f} | LR: {current_lr:.2e} | Speed: {speed:.2f} it/s")

    # প্রতি ৫০০ স্টেপ পরপর গুগুল ড্রাইভে সেভ
    if step % GPTConfig.save_interval == 0 or step == GPTConfig.max_iters:
        ckpt_path = os.path.join(DRIVE_CHECKPOINT_DIR, f"bengali_gpt_50m_step_{step}.pt")
        torch.save({
            'step': step,
            'model_state_dict': raw_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': accum_loss,
            'config': GPTConfig
        }, ckpt_path)
        print(f"💾 [SAVED] চেকপয়েন্ট ড্রাইভে সেভ হয়েছে: {ckpt_path}")

print("🎉 ৫০M মডেলের ফুল প্রি-ট্রেনিং সফলভাবে সম্পন্ন হয়েছে!")

In [ ]:
# Step 7: 🧪 মডেলের বাংলা লেখার ক্ষমতা টেস্ট করুন (ডিফল্ট প্রম্পটস)
eval_model = raw_model if 'raw_model' in locals() else model
eval_model.eval()

test_prompts = [
    "ডিজিটাল মার্কেটিং হলো",
    "ফেসবুক ও গুগলে বিজ্ঞাপন দিয়ে খুব সহজেই",
    "বাংলাদেশ একটি সুন্দর দেশ কারণ"
]

for prompt in test_prompts:
    enc = tokenizer.encode(prompt)
    ids = enc.ids if hasattr(enc, 'ids') else enc
    input_tensor = torch.tensor([ids], dtype=torch.long, device=device)
    
    with torch.no_grad():
        out = eval_model.generate(
            input_tensor,
            max_new_tokens=100,
            temperature=0.7,
            top_k=40,
            repetition_penalty=1.25
        )
    
    gen_text = tokenizer.decode(out[0].cpu().tolist())
    print("=" * 60)
    print(f"প্রম্পট: {prompt}")
    print(f"উত্তর: {gen_text}")

In [ ]:
# Step 8: 💬 আপনার নিজের কাস্টম প্রম্পট লিখে সরাসরি পরীক্ষা করুন!
# নিচের ইনপুট বক্সে আপনার যা ইচ্ছা বাংলায় লিখুন:
my_custom_prompt = "অনলাইনে ব্যবসা শুরু করতে হলে"  # <-- এখানে আপনার প্রশ্ন বা প্রম্পট লিখুন

enc = tokenizer.encode(my_custom_prompt)
ids = enc.ids if hasattr(enc, 'ids') else enc
input_tensor = torch.tensor([ids], dtype=torch.long, device=device)

eval_model = raw_model if 'raw_model' in locals() else model
eval_model.eval()

with torch.no_grad():
    out = eval_model.generate(
        input_tensor,
        max_new_tokens=150,        # উত্তরের দৈর্ঘ্য
        temperature=0.75,          # ক্রিয়েটিভিটি (0.7-0.8 আদর্শ)
        top_k=40,                  # অপ্রাসঙ্গিক শব্দ বাদ দেওয়া
        repetition_penalty=1.25    # পুনরাবৃত্তি ঠেকানো
    )

reply = tokenizer.decode(out[0].cpu().tolist())
print("=" * 60)
print("🤖 মডেলের উত্তর:")
print(reply)
print("=" * 60)